In [3]:
import random
from brian2 import *
import numpy as np
import matplotlib.pyplot as plt
import csv
import scipy
from tqdm import tqdm

In [4]:
chosen_neurons=np.random.randint(0,12500,50)
chosen_neurons

array([ 3390,  7349,  6788,  4133,  4829,  6720,  7117,  8561,  6588,
        8118,  9383,  8835,  4665,  4924, 10878,  7115,  5094,  8471,
        9281,  4056,  1025,  2397,  7339,   160,  6438,  8725, 10888,
        3569,  5354,  3112,  4657,  6257,   739,  8525,  6508,  2084,
          55,  2085,  5747,  6482,  5523,  2805, 10698,  1910, 11964,
        5580,  4562, 10349,  3337,   822])

In [23]:
def sim(trial, g, nu_ext_over_nu_thr, sim_time):
    """
    g -- relative inhibitory to excitatory synaptic strength
    nu_ext_over_nu_thr -- ratio of external stimulus rate to threshold rate
    sim_time -- simulation time
    """

    # network parameters
    N_E = 5000
    gamma = 0.25
    N_I = round(gamma * N_E)
    N = N_E + N_I
    epsilon = 0.1
    C_E = epsilon * N_E
    C_ext = C_E

    # neuron parameters
    tau = 20 * ms
    theta = 20 * mV
    V_r = 10 * mV
    tau_rp = 2 * ms

    # synapse parameters
    J = 0.1 * mV
    D = 1.5 * ms

    # external stimulus
    nu_thr = theta / (J * C_E * tau)

    defaultclock.dt = 0.1 * ms

    neurons = NeuronGroup(N,
                          """
                          dv/dt = -v/tau : volt (unless refractory)
                          """,
                          threshold="v > theta",
                          reset="v = V_r",
                          refractory=tau_rp,
                          method="exact",
    )

    excitatory_neurons = neurons[:N_E]
    inhibitory_neurons = neurons[N_E:]

    exc_synapses = Synapses(excitatory_neurons, target=neurons, on_pre="v += J", delay=D)
    exc_synapses.connect(p=epsilon)

    inhib_synapses = Synapses(inhibitory_neurons, target=neurons, on_pre="v += -g*J", delay=D)
    inhib_synapses.connect(p=epsilon)

    nu_ext = nu_ext_over_nu_thr * nu_thr

    external_poisson_input = PoissonInput(
        target=neurons, target_var="v", N=C_ext, rate=nu_ext, weight=J
    )

    rate_monitor = PopulationRateMonitor(neurons)

    #40 exc and 10 inh neurons

    spike_monitor_exc = SpikeMonitor(excitatory_neurons[:40])

    state_monitor_exc = StateMonitor(excitatory_neurons[:40], 'v', record=True, dt=1*ms)

    spike_monitor_inh = SpikeMonitor(inhibitory_neurons[:40])

    state_monitor_inh = StateMonitor(inhibitory_neurons[:40], 'v', record=True, dt=1*ms)

    run(sim_time, report='text')

    with open('data_for_michael/exc_spikes_brunel_trial{}.csv'.format(trial),'w') as f:
        writer=csv.writer(f)
        for it in range(len(spike_monitor_exc.t/ms)):
            writer.writerow([ spike_monitor_exc.t[it]/ms, spike_monitor_exc.i[it]])
    
    with open('data_for_michael/mempot_brunel_trial{}_exc.csv'.format(trial),'w') as f:
        writer=csv.writer(f)
        for it in range(len(state_monitor_exc.t/ms)):
            writer.writerow([ state_monitor_exc.v[:,it]/mV])

    with open('data_for_michael/inh_spikes_brunel_trial{}.csv'.format(trial),'w') as f:
        writer=csv.writer(f)
        for it in range(len(spike_monitor_inh.t/ms)):
            writer.writerow([ spike_monitor_inh.t[it]/ms, spike_monitor_inh.i[it]])
    
    with open('data_for_michael/mempot_brunel_trial{}_inh.csv'.format(trial),'w') as f:
        writer=csv.writer(f)
        for it in range(len(state_monitor_inh.t/ms)):
            writer.writerow([ state_monitor_inh.v[:,it]/mV])

In [25]:
params = {
        "g": 5,
        "nu_ext_over_nu_thr": 2,
        "simtime": 10000,
    }

for tr in tqdm(range(10)):
    sim(tr,
        params["g"],
        params["nu_ext_over_nu_thr"],
        params["simtime"] * ms,
    )

  0%|                                                    | 0/10 [00:00<?, ?it/s]

Starting simulation at t=0. s for a duration of 10. s
6.5707 s (65%) simulated in 10s, estimated 5s remaining.
10. s (100%) simulated in 15s


 10%|████▍                                       | 1/10 [00:26<03:55, 26.18s/it]

Starting simulation at t=0. s for a duration of 10. s
6.6434 s (66%) simulated in 10s, estimated 5s remaining.
10. s (100%) simulated in 15s


 20%|████████▊                                   | 2/10 [00:54<03:38, 27.25s/it]

...

 80%|███████████████████████████████████▏        | 8/10 [02:59<00:42, 21.29s/it]

Starting simulation at t=0. s for a duration of 10. s
6.277 s (62%) simulated in 10s, estimated 6s remaining.
10. s (100%) simulated in 15s


 90%|███████████████████████████████████████▌    | 9/10 [03:20<00:21, 21.13s/it]

Starting simulation at t=0. s for a duration of 10. s
6.2409 s (62%) simulated in 10s, estimated 6s remaining.
7.6929 s (76%) simulated in 7m 44s, estimated 2m 19s remaining.
10. s (100%) simulated in 7m 48s


100%|███████████████████████████████████████████| 10/10 [11:13<00:00, 67.33s/it]
